# 模型模块演示 (Models Module)

本notebook演示hscredit库中模型模块的全部功能，包含多种风控模型和评分卡。

In [1]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)
print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")

/Users/xiaoxi/anaconda3/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据形状: (12448, 85)

列名: ['MOB1', 'MOB2', 'loan_date', 'bj_qy24', 'mobtech80', 'bairong_v1', 'xiaoniu_c3', 'dxm_v6pro', 'bcf_fpv1', 'apply_for_admission_confidence', 'apply_for_admission_score', 'charging_fail_count_12m', 'charging_fail_count_1m', 'charging_fail_count_24m', 'charging_fail_count_3m', 'charging_fail_count_6m', 'consumer_finance_lender_count_12m', 'consumer_finance_lender_count_24m', 'consumer_finance_loan_confidence', 'consumer_finance_loan_credit_line', 'consumer_finance_loan_credit_line_avg', 'consumer_finance_loan_credit_line_max', 'consumer_finance_loan_lender_count', 'consumer_finance_loan_product_count', 'credit_loan_duration', 'hit_consumer_finance_lender_count', 'hit_lender_count', 'hit_network_loan_lender_count', 'last_performance_days', 'lender_count_12m', 'lender_count_1m', 'lender_count_24m', 'lender_count_3m', 'lender_count_6m', 'lender_inquiry_count', 'lender_inquiry_count_1m', 'lender_inquiry_count_3m', 'lender_inquiry_count_6m', 'loan_amt_between_1k_3k_coun

In [2]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")

# 准备数据
X = df[feature_cols]
y = df[target_col]

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"训练集: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"测试集: X_test={X_test.shape}, y_test={y_test.shape}")

特征数量: 80
训练集: X_train=(8713, 80), y_train=(8713,)
测试集: X_test=(3735, 80), y_test=(3735,)


## 1. 导入模型模块

In [3]:
from hscredit.core.models import (
    XGBoostRiskModel,
    LightGBMRiskModel,
    CatBoostRiskModel,
    LogisticRegression,
    ScoreCard
)
from hscredit.core.metrics import KS, AUC, Gini

print("所有模型已导入成功！")

所有模型已导入成功！


## 2. XGBoost风险模型

高效的梯度提升树模型，专为风控场景优化。支持早停（early_stopping）功能，适配XGBoost 2.0+版本的callbacks机制。

In [4]:
# XGBoost模型训练
xgb_model = XGBoostRiskModel(
    max_depth=5,
    learning_rate=0.1,
    n_estimators=100,
    eval_metric=['auc', 'logloss'],  # 多个评估指标
    early_stopping_rounds=10,       # 早停轮数
    verbose=False
)

xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
print("XGBoost模型训练完成！")

XGBoost模型训练完成！


In [5]:
# 模型预测
y_pred_proba = xgb_model.predict_proba(X_test)
y_pred = xgb_model.predict(X_test)

# 模型评估
xgb_auc: float = AUC(y_test, y_pred_proba[:, 1])
xgb_ks = KS(y_test, y_pred_proba[:, 1])
xgb_gini = Gini(y_test, y_pred_proba[:, 1])

print(f"XGBoost模型评估:")
print(f"  AUC: {xgb_auc:.4f}")
print(f"  KS: {xgb_ks:.4f}")
print(f"  Gini: {xgb_gini:.4f}")

XGBoost模型评估:
  AUC: 0.7220
  KS: 0.3163
  Gini: 0.4440


## 3. LightGBM风险模型

快速梯度提升树模型，适合大规模数据。支持早停（early_stopping）功能，适配LightGBM 4.0+版本的callbacks机制。

In [6]:
# LightGBM模型训练
lgb_model = LightGBMRiskModel(
    num_leaves=31,
    learning_rate=0.1,
    n_estimators=100,
    eval_metric=['auc', 'binary_logloss'],  # 多个评估指标
    early_stopping_rounds=10,               # 早停轮数
    first_metric_only=True,                 # 只用第一个指标进行早停
    verbose=False
)

lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
print("LightGBM模型训练完成！")

LightGBM模型训练完成！


In [7]:
# 模型预测和评估
y_pred_proba_lgb = lgb_model.predict_proba(X_test)
lgb_auc = AUC(y_test, y_pred_proba_lgb[:, 1])
lgb_ks = KS(y_test, y_pred_proba_lgb[:, 1])
lgb_gini = Gini(y_test, y_pred_proba_lgb[:, 1])

print(f"LightGBM模型评估:")
print(f"  AUC: {lgb_auc:.4f}")
print(f"  KS: {lgb_ks:.4f}")
print(f"  Gini: {lgb_gini:.4f}")

LightGBM模型评估:
  AUC: 0.7274
  KS: 0.3292
  Gini: 0.4547


## 4. CatBoost风险模型

对类别特征友好的提升树模型。支持早停（early_stopping）功能，CatBoost仍保留early_stopping_rounds参数（与XGBoost/LightGBM新版不同）。

In [8]:
# CatBoost模型训练
cat_model = CatBoostRiskModel(
    depth=6,
    learning_rate=0.1,
    iterations=100,
    eval_metric='AUC',
    early_stopping_rounds=10,  # 早停轮数
    verbose=False
)

cat_model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
print("CatBoost模型训练完成！")

CatBoost模型训练完成！


In [9]:
# 模型预测和评估
y_pred_proba_cat = cat_model.predict_proba(X_test)
cat_auc = AUC(y_test, y_pred_proba_cat[:, 1])
cat_ks = KS(y_test, y_pred_proba_cat[:, 1])
cat_gini = Gini(y_test, y_pred_proba_cat[:, 1])

print(f"CatBoost模型评估:")
print(f"  AUC: {cat_auc:.4f}")
print(f"  KS: {cat_ks:.4f}")
print(f"  Gini: {cat_gini:.4f}")

CatBoost模型评估:
  AUC: 0.7067
  KS: 0.3022
  Gini: 0.4135


## 5. 逻辑回归模型

扩展sklearn逻辑回归，支持统计信息和评分卡转换。

In [10]:
from hscredit.core.encoders import WOEEncoder
from hscredit.core.binning import OptimalBinning



binner = OptimalBinning(method='or_tools', prebinning_method='quantile', prebinning_params={'max_n_bins': 20})

# 先进行WOE编码
woe_encoder = WOEEncoder()
X_train_woe = woe_encoder.fit_transform(binner.fit_transform(X_train, y_train, metric="indices"), y_train)
X_test_woe = woe_encoder.transform(binner.transform(X_test, metric="indices"))

# 逻辑回归模型
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_woe, y_train)
print("逻辑回归模型训练完成！")

逻辑回归模型训练完成！


In [11]:
# 模型预测和评估
y_pred_proba_lr = lr_model.predict_proba(X_test_woe)
lr_auc = AUC(y_test, y_pred_proba_lr[:, 1])
lr_ks = KS(y_test, y_pred_proba_lr[:, 1])
lr_gini = Gini(y_test, y_pred_proba_lr[:, 1])

print(f"逻辑回归模型评估:")
print(f"  AUC: {lr_auc:.4f}")
print(f"  KS: {lr_ks:.4f}")
print(f"  Gini: {lr_gini:.4f}")

# 查看模型系数
print(f"\n模型系数（部分）:")
coef_df = pd.DataFrame({
    '特征': X_train_woe.columns[:10],
    '系数': lr_model.coef_[0][:10]
})
print(coef_df.to_string(index=False))

逻辑回归模型评估:
  AUC: 0.6407
  KS: 0.2149
  Gini: 0.2814

模型系数（部分）:
                            特征      系数
                       bj_qy24 -0.2591
                     mobtech80 -0.1428
                    bairong_v1 -0.5855
                    xiaoniu_c3 -0.5016
                     dxm_v6pro -0.3096
                      bcf_fpv1 -0.6679
apply_for_admission_confidence -0.3457
     apply_for_admission_score -0.2561
       charging_fail_count_12m -0.1576
        charging_fail_count_1m -0.6824


## 6. 评分卡模型

将逻辑回归模型转换为标准评分卡格式。

In [19]:
# 创建评分卡
scorecard = ScoreCard(
    # encoder=woe_encoder,  # 使用WOE编码器作为编码器
    binner=binner,
    # lr_model=lr_model,
    pdo=50,     # 每增加50分， odds翻倍
    rate=2,     # 基础倍率
    score=600,  # 基础分
    lower=300,  # 最低分
    upper=800,  # 最高分
)

print("评分卡模型创建成功！")

评分卡模型创建成功！


In [18]:
# 生成评分
scores = scorecard.predict(X_test_woe)
print(f"评分统计:")
print(f"  最小值: {scores.min():.2f}")
print(f"  最大值: {scores.max():.2f}")
print(f"  平均值: {scores.mean():.2f}")
print(f"  中位数: {np.median(scores):.2f}")
print(f"  标准差: {scores.std():.2f}")

AttributeError: 'NoneType' object has no attribute 'intercept_'

## 7. 模型对比

对比所有模型的性能。

In [ ]:
# 模型对比
model_results = pd.DataFrame({
    '模型': ['XGBoost', 'LightGBM', 'CatBoost', '逻辑回归'],
    'AUC': [xgb_auc, lgb_auc, cat_auc, lr_auc],
    'KS': [xgb_ks, lgb_ks, cat_ks, lr_ks],
    'Gini': [xgb_gini, lgb_gini, cat_gini, lr_gini]
})

model_results = model_results.sort_values('AUC', ascending=False)
print("模型性能对比:")
print(model_results.to_string(index=False))

## 8. 保存模型结果

将训练好的模型和结果保存到文件。

In [ ]:
from hscredit.utils import save_pickle

# 保存模型
model_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/models'
os.makedirs(model_path, exist_ok=True)

save_pickle(xgb_model, os.path.join(model_path, 'xgb_model.pkl'))
save_pickle(lgb_model, os.path.join(model_path, 'lgb_model.pkl'))
save_pickle(cat_model, os.path.join(model_path, 'cat_model.pkl'))
save_pickle(lr_model, os.path.join(model_path, 'lr_model.pkl'))
save_pickle(woe_encoder, os.path.join(model_path, 'woe_encoder.pkl'))

# 保存结果到Excel
output_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/model_results.xlsx'
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    model_results.to_excel(writer, sheet_name='模型对比', index=False)
    pd.DataFrame({'评分': scores}).to_excel(writer, sheet_name='评分分布', index=False)

print(f"模型和结果已保存至: {model_path}")
print(f"Excel结果已保存至: {output_path}")